In [73]:
import requests

from dotenv import load_dotenv
from pydantic import BaseModel

from agents import (
    Agent,
    Runner,
    function_tool,
    GuardrailFunctionOutput,
    RunContextWrapper,
    TResponseInputItem,
    InputGuardrailTripwireTriggered,
    OutputGuardrailTripwireTriggered,
    OpenAIChatCompletionsModel
)

from agents.decorators import input_guardrail, output_guardrail
import os
from IPython.display import Markdown,display
from openai import AsyncOpenAI

In [74]:
load_dotenv(override=True)

True

In [75]:
api_key=os.getenv("GEMINI_API_KEY_2")


In [76]:
client=AsyncOpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [77]:
model=OpenAIChatCompletionsModel(
    model="gemini-flash-latest",
    openai_client=client
)

In [78]:
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
PUSHOVER_USER_KEY = os.getenv("PUSHOVER_USER")
PUSHOVER_API_TOKEN = os.getenv("PUSHOVER_TOKEN")

In [79]:
@function_tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""

    api_key = os.getenv("OPENWEATHER_API_KEY")

    if not api_key:
        return "OPENWEATHER_API_KEY is not configured."

    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": city,
        "appid": api_key,
        "units": "metric",
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()

        data = response.json()

        temperature = data["main"]["temp"]
        feels_like = data["main"]["feels_like"]
        humidity = data["main"]["humidity"]
        description = data["weather"][0]["description"]

        return (
            f"Weather in {city}:\n"
            f"Temperature: {temperature}°C\n"
            f"Feels like: {feels_like}°C\n"
            f"Condition: {description}\n"
            f"Humidity: {humidity}%"
        )

    except requests.exceptions.HTTPError:
        return f"Could not find weather data for {city}."

    except requests.exceptions.RequestException:
        return "Weather service is currently unavailable."

In [80]:
@function_tool
def send_notification(message: str) -> str:
    """Send a notification through Pushover."""

    url = "https://api.pushover.net/1/messages.json"

    data = {
        "token": PUSHOVER_API_TOKEN,
        "user": PUSHOVER_USER_KEY,
        "message": message,
    }

    response = requests.post(url, data=data, timeout=10)

    if response.status_code == 200:
        return "Notification sent successfully."

    return "Failed to send notification."

In [81]:
class InputGuardrailOutput(BaseModel):
    is_weather_related: bool
    reasoning: str


In [82]:
input_guardrail_agent = Agent(
    name="Weather Input Guardrail",
    instructions="""
    Determine whether the user's request is related to weather.

    Weather-related requests include:
    - Current weather
    - Temperature
    - Humidity
    - Rain
    - Wind
    - Weather forecast
    - Weather updates
    - Sending a weather notification

    Return is_weather_related=True only when the request
    is related to weather or a weather notification.
    """,
    output_type=InputGuardrailOutput,
    model=model
)

In [83]:
@input_guardrail(run_in_parallel=False)
async def weather_input_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:

    result = await Runner.run(
        input_guardrail_agent,
        input,
        context=ctx.context,
    )

    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=not result.final_output.is_weather_related,
    )


In [84]:
class OutputGuardrailOutput(BaseModel):
    is_valid_weather_response: bool
    reasoning: str

In [85]:
output_guardrail_agent = Agent(
    name="Weather Output Guardrail",
    instructions="""
    Check whether the assistant's response is a valid weather-related
    response.

    The response should:
    - Be related to weather.
    - Not contain unrelated information.
    - Not expose API keys, tokens, or secrets.
    - Be clear and useful to the user.

    Return is_valid_weather_response=True when the response is valid.
    """,
    output_type=OutputGuardrailOutput,
    model=model
)

In [86]:
@output_guardrail
async def weather_output_guardrail(
    ctx: RunContextWrapper,
    agent: Agent,
    output: str,
) -> GuardrailFunctionOutput:

    result = await Runner.run(
        output_guardrail_agent,
        output,
        context=ctx.context,
    )

    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=not result.final_output.is_valid_weather_response,
    )

In [87]:
weather_agent = Agent(
    name="Weather Agent",

    instructions="""
    You are a professional Weather Agent.

    Your responsibilities:

    1. Answer only weather-related questions.
    2. Use get_weather whenever the user asks for current weather.
    3. Never guess weather information.
    4. Use the exact city provided by the user.
    5. Use send_notification ONLY when the user explicitly asks
       you to send a notification.
    6. If the user asks for weather but does not ask for a notification,
       only provide the weather information.
    7. If the user asks for a notification, first get the weather
       information and then send the notification.
    8. Never reveal API keys, tokens, or internal implementation details.
    9. Keep responses simple and concise.

    Examples:

    User: "What is the weather in Lahore?"
    → Call get_weather.

    User: "Tell me the weather in Islamabad and send me a notification."
    → Call get_weather first.
    → Then call send_notification.

    User: "Send me a notification about the weather in Karachi."
    → Call get_weather first.
    → Then call send_notification.

    User: "Who is the president of Pakistan?"
    → This is not a weather request.
    → Do not answer it.
    """,

    tools=[
        get_weather,
        send_notification,
    ],

    input_guardrails=[
        weather_input_guardrail,
    ],

    output_guardrails=[
        weather_output_guardrail,
    ],
    model=model
)

In [88]:


user_input = "You: send notification about lahore weather"



result = await Runner.run(
    weather_agent,
    user_input,
)





OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


In [90]:
result.final_output

'The notification about the weather in Lahore (32.92°C, few clouds, humidity 57%) has been sent successfully.'

OPENAI_API_KEY is not set, skipping trace export
